# 04 — Feature engineering & second-pass modeling

**Phase:** After a **leakage-safe baseline** (notebook 03), this notebook runs a **realistic second iteration**:
enrich the **tabular representation** of each movie with **business-structured** features, then **re-train the same model family**
on the expanded matrix and **compare honestly** to the baseline feature set.

**Workflow:** business rationale for features → load processed data → engineer signals → **leakage audit** →
feature-set comparison → **same train/test discipline** as notebook 03 → baseline vs enriched metrics →
interpretation → executive conclusions → controlled next steps.

**Style:** each analytical block ends with **Observation → Business interpretation → Modeling implication**
(consistent with notebooks 01–03).

**Setting:** **Pre-release** prediction only — nothing derived from realized box office or post-release engagement enters `X`.


## 1. Business goal of feature engineering

Notebook 03 established a **defensible baseline**: out-of-sample performance was typically **weak-to-moderate** — useful for **triage and discussion**, not for pretending the problem is solved.

In professional ML engagements, **signal quality** (features + target + evaluation protocol) usually dominates marginal algorithm tweaks. Here we inject **more business structure** into `X` so models can represent:

- **Production scale** (skewed budgets, company footprint, international footprint, language breadth),
- **Positioning proxies** (runtime bands as “compact” vs “event” programming),
- **Release timing** (human-readable **seasonality**, not only integer months),
- **Content complexity** (how many genres are blended into one title).

We are explicitly **not** doing “Kaggle-style feature spam”: every addition should survive an **oral defense** as something a studio analyst could explain **before release**.

---

**Observation:** First-pass models often under-use heavy tails and calendar structure.  
**Business interpretation:** Stakeholders reason in **bands** (budget magnitude, season, slate complexity).  
**Modeling implication:** Re-run the **same** estimators and metrics so lift (or lack of it) attributes to **representation**, not optimizer luck.


## 2. Load processed dataset

We reuse:

`data/processed/movies_cleaned_with_target.csv`

**Defensive behavior:** resolve `PROJECT_ROOT` whether the notebook runs from repo root or `notebooks/`, print **path + shape**, and keep the same **forbidden-in-X** contract as notebook 03.

---

**Observation:** Versioned processed data keeps iterations comparable.  
**Business interpretation:** Feature iterations should not silently change the commercial outcome definition.  
**Modeling implication:** Re-audit leakage whenever `X` gains columns.


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

_CWD = Path.cwd().resolve()
if (_CWD / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD
    print("Warning: data/processed not found; using cwd as PROJECT_ROOT.")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "movies_cleaned_with_target.csv"
PLOTS_DIR = PROJECT_ROOT / "plots" / "modeling"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk", font_scale=0.95)
plt.rcParams["figure.figsize"] = (11, 5.5)
plt.rcParams["axes.titlesize"] = 14


def save_fig(name: str) -> Path:
    """Save current figure to plots/modeling/ and close it (avoids memory buildup)."""
    path = PLOTS_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.close()
    return path


TARGET_COLUMN = "movie_success_class"
FORBIDDEN_IN_X = {"revenue", "roi", "log_roi", TARGET_COLUMN}
LEAKAGE_DROP = ["revenue", "roi", "log_roi", TARGET_COLUMN]

BASELINE_CANDIDATES = [
    "budget",
    "runtime",
    "main_genre",
    "original_language",
    "release_month",
    "release_quarter",
    "genre_count",
    "production_company_count",
    "production_country_count",
    "spoken_language_count",
]

if not DATA_PATH.exists():
    df = None
    print(f"Missing dataset: {DATA_PATH}")
    print("Run notebooks/01_dataset_audit_and_target.ipynb first.")
else:
    df = pd.read_csv(DATA_PATH)
    print("Loaded:", DATA_PATH, "| shape:", df.shape)

if df is not None:
    print("TARGET_COLUMN:", TARGET_COLUMN)
    print("Leakage columns present in file (audit only):", [c for c in LEAKAGE_DROP if c in df.columns])
else:
    print("No dataframe loaded.")


## 3. Create engineered features (pre-release, leakage-safe)

We build **`df_fe`** as a copy of the processed frame and add interpretable columns. **Forbidden fields are never used as inputs.**

**Feature definitions**

- **`budget_log`:** `log1p(budget)` — budgets are extremely right-skewed; a log transform stabilizes scale for linear models and splits in tree models.
- **`runtime_bucket`:** `short` (&lt;90 min), `medium` (90–120), `long` (&gt;120) — coarse proxy for “programming length” vs “tentpole runtime.”
- **`production_scale`:** `indie` / `mid_scale` / `large_scale` from **tertiles of `production_company_count` fit on the training split only** (labels applied to train and test after the split — avoids leaking test distribution into thresholds).
- **`international_production`:** `1` if `production_country_count` &gt; 1 else `0` — co-production / international footprint (NaNs treated as 0 footprint before comparison).
- **`multilingual_movie`:** `1` if `spoken_language_count` &gt; 1 else `0`.
- **`release_season`:** `winter` / `spring` / `summer` / `fall` from `release_month`.
- **`genre_complexity`:** `focused` (1 genre), `mixed` (2–3), `hybrid` (4+) on `genre_count`.
- **`decade`:** e.g. `2010s` from `release_year` when present; otherwise documented warning and `__missing__`.

A temporary placeholder **`__pending__`** is used for `production_scale` until the train/test partition is fixed; it is overwritten before any model is fit.

---

**Observation:** Most engineered fields are **monotone compressions** or **named buckets** of information already in the baseline list.  
**Business interpretation:** We are encoding **how executives cluster reality**, not inventing post-release facts.  
**Modeling implication:** Expect wider one-hot matrices — monitor whether lift justifies sparsity.


In [ ]:
STATIC_ENGINEERED = [
    "budget_log",
    "runtime_bucket",
    "international_production",
    "multilingual_movie",
    "release_season",
    "genre_complexity",
    "decade",
]

if df is None:
    df_fe = None
    print("Skip feature engineering (no data).")
else:
    df_fe = df.copy()

    df_fe["budget_log"] = np.log1p(df_fe["budget"].clip(lower=0))

    def _runtime_bucket(v) -> str:
        if pd.isna(v):
            return "__missing__"
        r = float(v)
        if r < 90:
            return "short"
        if r <= 120:
            return "medium"
        return "long"

    df_fe["runtime_bucket"] = df_fe["runtime"].map(_runtime_bucket)

    df_fe["international_production"] = (df_fe["production_country_count"].fillna(0) > 1).astype(int)
    df_fe["multilingual_movie"] = (df_fe["spoken_language_count"].fillna(0) > 1).astype(int)

    def _season_from_month(m) -> str:
        if pd.isna(m):
            return "__missing__"
        mi = int(m)
        if mi in (12, 1, 2):
            return "winter"
        if mi in (3, 4, 5):
            return "spring"
        if mi in (6, 7, 8):
            return "summer"
        if mi in (9, 10, 11):
            return "fall"
        return "__missing__"

    if "release_month" in df_fe.columns:
        df_fe["release_season"] = df_fe["release_month"].map(_season_from_month)
    else:
        df_fe["release_season"] = "__missing__"

    def _genre_complexity(gc) -> str:
        if pd.isna(gc):
            return "__missing__"
        g = int(gc)
        if g <= 1:
            return "focused"
        if g <= 3:
            return "mixed"
        return "hybrid"

    df_fe["genre_complexity"] = df_fe["genre_count"].map(_genre_complexity)

    if "release_year" in df_fe.columns:
        yr = pd.to_numeric(df_fe["release_year"], errors="coerce")

        def _decade(y):
            if pd.isna(y):
                return "__missing__"
            yi = int(y)
            d = (yi // 10) * 10
            return f"{d}s"

        df_fe["decade"] = yr.map(_decade)
        if (df_fe["decade"] == "__missing__").all():
            print("Warning: `decade` is all missing — check `release_year` quality.")
    else:
        df_fe["decade"] = "__missing__"
        print("Warning: column `release_year` not found — `decade` set to `__missing__`.")

    df_fe["production_scale"] = "__pending__"

    print("Static engineered columns:", STATIC_ENGINEERED)
    print("df_fe shape:", df_fe.shape)


## 4. Leakage audit after engineering

We restate the **modeling contract** after transforms:

- Print **`FORBIDDEN_IN_X`**.
- Build **`BASELINE_FEATURES`** (notebook 03 contract) and **`ENRICHED_FEATURES`** = baseline + engineered columns that exist.
- **`assert`** the target and forbidden fields are absent from both feature lists.
- Print **feature counts** for baseline vs enriched.

---

**Observation:** Leakage is often accidental when new columns are merged from other tables.  
**Business interpretation:** A visible audit trail is part of **professional credibility**.  
**Modeling implication:** Assertions are cheap compared to explaining inflated metrics later.


In [ ]:
ENGINEERED_FOR_MODEL = STATIC_ENGINEERED + ["production_scale"]

if df is None or df_fe is None:
    BASELINE_FEATURES = []
    ENRICHED_FEATURES = []
    print("Skip leakage audit (no data).")
else:
    print("\n===== LEAKAGE AUDIT (POST-ENGINEERING) =====")
    print("FORBIDDEN_IN_X (must never appear in X):", sorted(FORBIDDEN_IN_X))

    present_base = [c for c in BASELINE_CANDIDATES if c in df_fe.columns]
    missing_base = [c for c in BASELINE_CANDIDATES if c not in df_fe.columns]
    if missing_base:
        print("Skipping missing baseline columns:", missing_base)

    BASELINE_FEATURES = [c for c in present_base if c not in FORBIDDEN_IN_X]
    eng_ok = [c for c in ENGINEERED_FOR_MODEL if c in df_fe.columns]
    ENRICHED_FEATURES = BASELINE_FEATURES + eng_ok

    assert TARGET_COLUMN not in BASELINE_FEATURES, "Leakage guard: target in baseline features."
    assert TARGET_COLUMN not in ENRICHED_FEATURES, "Leakage guard: target in enriched features."
    leak_b = set(BASELINE_FEATURES) & FORBIDDEN_IN_X
    leak_e = set(ENRICHED_FEATURES) & FORBIDDEN_IN_X
    assert not leak_b, f"Forbidden columns in baseline features: {sorted(leak_b)}"
    assert not leak_e, f"Forbidden columns in enriched features: {sorted(leak_e)}"

    print("n_features baseline (leakage-safe):", len(BASELINE_FEATURES))
    print("n_features enriched (leakage-safe):", len(ENRICHED_FEATURES))
    print("BASELINE_FEATURES:", BASELINE_FEATURES)
    print("ENRICHED_FEATURES:", ENRICHED_FEATURES)


## 5. Compare original vs engineered feature sets

A compact **summary table** answers: how much wider did the design become, and how did the **numeric vs categorical** split evolve? (One-hot cardinality will grow with new categorical levels.)

---

**Observation:** Width is a **cost** (variance, interpretability, training time) as well as a potential benefit.  
**Business interpretation:** Scope control matters for production PoCs.  
**Modeling implication:** If lift is flat, prefer **parsimony** until new domain data arrives.


In [ ]:
def _count_num_cat(feature_cols: list[str], cat_names: list[str]) -> tuple[int, int]:
    cats = [c for c in feature_cols if c in cat_names]
    nums = [c for c in feature_cols if c not in cats]
    return len(nums), len(cats)


if df is None or df_fe is None or not BASELINE_FEATURES:
    feature_summary_df = None
    print("Skip feature comparison.")
else:
    baseline_cat_names = [
        "main_genre",
        "original_language",
        "release_month",
        "release_quarter",
    ]
    extra_cat_names = [
        "runtime_bucket",
        "production_scale",
        "release_season",
        "genre_complexity",
        "decade",
    ]
    enriched_cat_names = baseline_cat_names + [c for c in extra_cat_names if c in ENRICHED_FEATURES]

    bn, bc = _count_num_cat(BASELINE_FEATURES, baseline_cat_names)
    en, ec = _count_num_cat(ENRICHED_FEATURES, enriched_cat_names)

    feature_summary_df = pd.DataFrame(
        [
            {
                "setting": "baseline (notebook 03 contract)",
                "n_total_features": len(BASELINE_FEATURES),
                "n_numeric": bn,
                "n_categorical": bc,
            },
            {
                "setting": "enriched (baseline + engineered)",
                "n_total_features": len(ENRICHED_FEATURES),
                "n_numeric": en,
                "n_categorical": ec,
            },
        ]
    )
    print("\n===== FEATURE SET SUMMARY =====")
    display(feature_summary_df)


## 6. Re-train models: baseline vs enriched

**Same philosophy as notebook 03:** `ColumnTransformer` + `Pipeline`, **median / most_frequent imputers**, `StandardScaler` on numerics, `OneHotEncoder(handle_unknown="ignore")` on categoricals.

**Models:** multinomial **LogisticRegression** (`class_weight="balanced"`), **RandomForestClassifier** (`class_weight="balanced"`), **GradientBoostingClassifier**, and **XGBoost** if importable.

**Split:** `test_size=0.2`, `random_state=42`, **`stratify=y`** — run **twice** on different `X` matrices; with identical `y` and `random_state`, sklearn assigns the **same** train/test rows to each setting.

**Primary comparison table**

| model | baseline_macro_f1 | engineered_macro_f1 | delta |

(Accuracy deltas are printed alongside for context — macro-F1 remains the headline metric for imbalanced three-way outcomes.)

---

**Observation:** A controlled A/B on **features** isolates the value of domain encoding.  
**Business interpretation:** This mirrors how delivery teams attribute lift in **iteration reviews**.  
**Modeling implication:** If trees gain more than logistic regression, non-linear interactions or thresholds may matter; if nothing moves, the bottleneck is likely **missing drivers** (marketing, IP, talent).


In [ ]:
HAS_XGB = False
try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except Exception:
    XGBClassifier = None
    print("xgboost not available — skipping XGBClassifier.")


def prepare_X(feature_cols: list[str], frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    X = frame[feature_cols].copy()
    baseline_cat_names = [
        "main_genre",
        "original_language",
        "release_month",
        "release_quarter",
    ]
    extra_cat_names = [
        "runtime_bucket",
        "production_scale",
        "release_season",
        "genre_complexity",
        "decade",
    ]
    cat_cols = [c for c in baseline_cat_names + extra_cat_names if c in X.columns]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    for col in ["main_genre", "original_language"]:
        if col in X.columns:
            X[col] = X[col].astype("string").fillna("__missing__")

    for col in ["release_month", "release_quarter"]:
        if col in X.columns:
            X[col] = X[col].apply(lambda v: "__missing__" if pd.isna(v) else str(int(v)))

    for col in cat_cols:
        X[col] = X[col].astype(str)

    for col in cat_cols:
        n_u = X[col].nunique(dropna=False)
        if n_u > 200:
            print(
                f"WARNING: categorical `{col}` has {n_u} unique values (>200). "
                "Consider grouping later; OHE may become very wide."
            )

    return X, num_cols, cat_cols


def build_preprocessor(num_cols: list[str], cat_cols: list[str]) -> ColumnTransformer:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    transformers = []
    if num_cols:
        transformers.append(("num", numeric_transformer, num_cols))
    if cat_cols:
        transformers.append(("cat", categorical_transformer, cat_cols))
    if not transformers:
        raise ValueError("No numeric or categorical columns to preprocess.")
    return ColumnTransformer(transformers=transformers)


def train_suite(
    X_train,
    X_test,
    y_train,
    y_test,
    num_cols: list[str],
    cat_cols: list[str],
    banner: str,
):
    def make_preprocessor():
        return build_preprocessor(num_cols, cat_cols)

    def make_lr():
        return Pipeline(
            steps=[
                ("prep", make_preprocessor()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=42,
                        solver="lbfgs",
                    ),
                ),
            ]
        )

    def make_rf():
        return Pipeline(
            steps=[
                ("prep", make_preprocessor()),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=200,
                        class_weight="balanced",
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        )

    def make_gbc():
        return Pipeline(
            steps=[
                ("prep", make_preprocessor()),
                ("clf", GradientBoostingClassifier(random_state=42)),
            ]
        )

    builders = {
        "logistic_regression": make_lr,
        "random_forest": make_rf,
        "gradient_boosting": make_gbc,
    }
    if HAS_XGB:

        def make_xgb():
            return Pipeline(
                steps=[
                    ("prep", make_preprocessor()),
                    (
                        "clf",
                        XGBClassifier(
                            n_estimators=200,
                            max_depth=5,
                            learning_rate=0.1,
                            objective="multi:softprob",
                            num_class=int(y_train.nunique()),
                            random_state=42,
                            n_jobs=-1,
                            eval_metric="mlogloss",
                        ),
                    ),
                ]
            )

        builders["xgboost"] = make_xgb

    rows = []
    models: dict = {}
    print(f"\n===== {banner} =====")
    for name, b in builders.items():
        pipe = b()
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        macro = f1_score(y_test, y_pred, average="macro")
        rows.append({"model": name, "accuracy": acc, "macro_f1": macro})
        models[name] = pipe
        print(name, "| accuracy:", round(acc, 3), "| macro_f1:", round(macro, 3))
    return pd.DataFrame(rows), models


def production_scale_from_train(train_counts: pd.Series, apply_counts: pd.Series) -> pd.Series:
    t = pd.to_numeric(train_counts, errors="coerce").dropna()
    if t.empty:
        q1, q2 = 0.0, 1.0
    else:
        q1, q2 = t.quantile(1 / 3), t.quantile(2 / 3)
    if q1 == q2:
        q2 = q2 + 1e-6

    def bucket(x):
        if pd.isna(x):
            return "__missing__"
        xv = float(x)
        if xv <= q1:
            return "indie"
        if xv <= q2:
            return "mid_scale"
        return "large_scale"

    return apply_counts.map(bucket)


if df is None or df_fe is None or not ENRICHED_FEATURES:
    y = None
    comparison_df = None
    fitted_enriched = {}
    X_test_e = None
    y_train = y_test = None
    print("Skip modeling (no data or empty feature list).")
else:
    y = df_fe[TARGET_COLUMN].astype(str)

    X_b_full, num_b, cat_b = prepare_X(BASELINE_FEATURES, df_fe)
    X_e_full, num_e, cat_e = prepare_X(ENRICHED_FEATURES, df_fe)

    print("\n===== TRAIN / TEST SPLIT (shared y, stratify=y) =====")
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_b_full,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )
    X_train_e, X_test_e, _, _ = train_test_split(
        X_e_full,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    if "production_company_count" in X_train_e.columns:
        X_train_e = X_train_e.copy()
        X_test_e = X_test_e.copy()
        tr_ref = X_train_e["production_company_count"]
        X_train_e["production_scale"] = production_scale_from_train(tr_ref, X_train_e["production_company_count"])
        X_test_e["production_scale"] = production_scale_from_train(tr_ref, X_test_e["production_company_count"])
        X_train_e["production_scale"] = X_train_e["production_scale"].astype(str)
        X_test_e["production_scale"] = X_test_e["production_scale"].astype(str)
    else:
        X_train_e["production_scale"] = "__missing__"
        X_test_e["production_scale"] = "__missing__"

    print("X_train baseline:", X_train_b.shape, "| X_test baseline:", X_test_b.shape)
    print("X_train enriched:", X_train_e.shape, "| X_test enriched:", X_test_e.shape)
    print("Train class distribution (%):")
    display((y_train.value_counts(normalize=True) * 100).round(1).to_frame("train_%"))
    print("Test class distribution (%):")
    display((y_test.value_counts(normalize=True) * 100).round(1).to_frame("test_%"))

    results_baseline, fitted_baseline = train_suite(
        X_train_b,
        X_test_b,
        y_train,
        y_test,
        num_b,
        cat_b,
        "MODEL TRAINING — BASELINE FEATURES",
    )
    results_enriched, fitted_enriched = train_suite(
        X_train_e,
        X_test_e,
        y_train,
        y_test,
        num_e,
        cat_e,
        "MODEL TRAINING — ENRICHED FEATURES",
    )

    cmp = results_baseline.rename(
        columns={"macro_f1": "baseline_macro_f1", "accuracy": "baseline_accuracy"}
    ).merge(
        results_enriched.rename(
            columns={"macro_f1": "engineered_macro_f1", "accuracy": "engineered_accuracy"}
        ),
        on="model",
    )
    cmp["delta_macro_f1"] = cmp["engineered_macro_f1"] - cmp["baseline_macro_f1"]
    cmp["delta_accuracy"] = cmp["engineered_accuracy"] - cmp["baseline_accuracy"]
    cmp = cmp.sort_values("engineered_macro_f1", ascending=False).reset_index(drop=True)
    comparison_df = cmp.copy()

    print("\n===== BASELINE vs ENRICHED =====")
    display(
        cmp[
            [
                "model",
                "baseline_macro_f1",
                "engineered_macro_f1",
                "delta_macro_f1",
                "baseline_accuracy",
                "engineered_accuracy",
                "delta_accuracy",
            ]
        ].round(3)
    )


## 7. Performance read — did feature engineering help?

**How to read this iteration (consulting tone, no hype):**

- **Macro-F1** still averages the three outcome buckets with equal weight — a few points of movement **may** or **may not** be material depending on baseline noise and class support.
- If **tree models** improve more than **logistic regression**, that suggests **threshold / interaction** structure the linear model cannot carve cleanly (or simply more capacity absorbing noise — always sanity-check with simplicity).
- If **accuracy rises but macro-F1 falls**, the model may be leaning on the **majority class** — usually a red flag for portfolio thinking.
- If **nothing moves**, the honest story is often **structural**: the PoC is missing marketing intensity, franchise/IP, talent, competition — not “wrong log transform.”

---

**Observation:** Feature iterations should be judged on **macro-behaviour across classes**, not a single headline number.  
**Business interpretation:** Small lifts can still **reshape rankings** for borderline projects even when accuracy looks flat.  
**Modeling implication:** Next gains may require **new data contracts** (external signals) more than more bucketing.


## 8. Feature importance (enriched tree model)

We plot **transformed** importances for the **best tree-based** model on the **enriched** matrix (among Random Forest, Gradient Boosting, XGBoost by **macro-F1** on the test split).

**Questions this view answers**

- Do **`budget_log`**, **`production_scale`**, or **season** / **decade** dimensions show up near the top?
- Are gains **concentrated** in a few interpretable directions, or **diffuse** (possible noise absorption)?

If extraction fails (API mismatch, rare sklearn edge case), we print a **short warning** and continue — the notebook should still run end-to-end.

---

**Observation:** Importance is **not causal** — it ranks split utility in this dataset.  
**Business interpretation:** Use it to **prompt questions** (“why is season showing up?”), not to assert mandates.  
**Modeling implication:** Pair importance with **error analysis** and **holdout stability** before production narratives.


In [ ]:
if comparison_df is None or not len(comparison_df):
    print("Skip plots (no comparison results).")
else:
    print("\n===== EVALUATION & FIGURES =====")

    melt_cols = ["model", "baseline_macro_f1", "engineered_macro_f1"]
    long_df = comparison_df[melt_cols].melt(
        id_vars="model",
        value_vars=["baseline_macro_f1", "engineered_macro_f1"],
        var_name="setting",
        value_name="macro_f1",
    )
    long_df["setting"] = long_df["setting"].map(
        {"baseline_macro_f1": "baseline", "engineered_macro_f1": "enriched"}
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=long_df, x="model", y="macro_f1", hue="setting", palette="deep", ax=ax)
    ax.set_title("Macro-F1: baseline vs enriched features (test)")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    save_fig("04_fe_macrof1_baseline_vs_enriched")

    best_enriched_name = comparison_df.sort_values("engineered_macro_f1", ascending=False).iloc[0]["model"]
    best_pipe = fitted_enriched.get(best_enriched_name)
    if best_pipe is None:
        print("No fitted enriched model available for confusion matrix.")
    else:
        y_hat = best_pipe.predict(X_test_e)
        print("\nBest enriched model (by macro-F1):", best_enriched_name)
        print("\nClassification report (enriched, test):")
        print(classification_report(y_test, y_hat, digits=3))

        preferred = ["flop", "average", "hit"]
        labels = [c for c in preferred if c in set(y_test) | set(y_hat)]
        labels += sorted((set(y_test) | set(y_hat)) - set(labels))
        cm = confusion_matrix(y_test, y_hat, labels=labels)
        fig, ax = plt.subplots(figsize=(7, 5.5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title(f"Confusion matrix — enriched — {best_enriched_name}")
        save_fig("05_fe_confusion_matrix_best_enriched")

    print("\n===== FEATURE IMPORTANCE (ENRICHED TREE MODELS) =====")
    tree_candidates = [m for m in ["random_forest", "gradient_boosting", "xgboost"] if m in fitted_enriched]
    tree_rank = comparison_df[comparison_df["model"].isin(tree_candidates)].sort_values(
        "engineered_macro_f1", ascending=False
    )
    if tree_rank.empty:
        print("No tree models in enriched run — skip importance.")
    else:
        best_tree = tree_rank.iloc[0]["model"]
        tree_pipe = fitted_enriched[best_tree]
        clf = tree_pipe.named_steps["clf"]
        prep = tree_pipe.named_steps["prep"]
        try:
            names = prep.get_feature_names_out()
            if not hasattr(clf, "feature_importances_"):
                print(f"Warning: `{best_tree}` has no `feature_importances_`; skip plot.")
            else:
                imp = np.asarray(clf.feature_importances_)
                if len(names) != len(imp):
                    print(
                        "Warning: length mismatch between feature names and importances; skip plot."
                    )
                else:
                    eng_tokens = (
                        "budget_log",
                        "runtime_bucket",
                        "production_scale",
                        "international_production",
                        "multilingual_movie",
                        "release_season",
                        "genre_complexity",
                        "decade",
                    )
                    fi = pd.Series(imp, index=names).sort_values(ascending=False)
                    top = fi.head(20)
                    fig, ax = plt.subplots(figsize=(9, 7))
                    ax.barh(top.index.astype(str), top.values, color="#276749", edgecolor="white", linewidth=0.5)
                    ax.invert_yaxis()
                    ax.set_title(f"Top 20 importances — enriched — {best_tree}")
                    ax.set_xlabel("Importance")
                    save_fig("06_fe_feature_importance_top20")
                    display(top.to_frame("importance").round(4))
                    hits = [n for n in top.index if any(tok in str(n) for tok in eng_tokens)]
                    print("Engineered-signal hits in top-20 (name fragments):", hits[:12])
        except Exception as exc:
            print("Warning: could not extract or plot feature importances:", exc)


## 9. Business conclusions

**What this iteration demonstrates (even when lift is small):**

- **Richer, named representations** of budget skew, slate complexity, timing, and production footprint are how analytics teams **encode domain** for models — the right baseline for a studio PoC.
- **Iteration discipline** matters as much as the numbers: same split, same metrics, explicit leakage audit — that is what makes the story **client-ready**.
- **Pre-release prediction remains uncertain:** TMDB tabular fields do not observe **marketing intensity, competition, talent/IP, or word-of-mouth** — models will **plateau** until the data contract expands.

**Executive takeaway:** treat output as **ranking / triage support** with **human-in-the-loop** decisions, not autonomous greenlights.

---

**Observation:** Performance ceilings are often set by **unobserved drivers**, not optimizer choice.  
**Business interpretation:** The value proposition is **transparent prioritization**, not oracle accuracy.  
**Modeling implication:** Invest next in **data acquisition** and **time-based validation** before deeper model complexity.


## 10. Next improvements (controlled backlog)

Practical, **defensible** extensions that usually beat “more default parameters”:

- **Cast / director / producer track records** (requires entity resolution + external career metrics).
- **Franchise / sequel / IP indicators** (binary or tiered buckets from title patterns + curated lists).
- **NLP on synopsis** (embeddings or sparse TF–IDF) if stakeholders accept text scope and governance.
- **Inflation-adjusted budget** (CPI index by country/year) for cross-era comparability.
- **External calendars** (competing wide releases, sports events, holidays).
- **Time-based split** (train on older releases, validate on newer) to reduce optimistic bias from era effects.
- **Calibration** (`CalibratedClassifierCV`) so probabilities read credibly in finance conversations.

---

**Observation:** The strongest roadmaps alternate **new information** with **evaluation hygiene**.  
**Business interpretation:** Each item should map to a **decision** (greenlight tier, marketing spend band).  
**Modeling implication:** Version the processed dataset whenever the feature contract changes materially.
